# Creating CTT (Text-Grammar) files from BHSA passages

## Loading the Workbench

In [1]:
import sys, os, collections
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt; plt.rcdefaults()
from matplotlib.pyplot import figure
from collections import Counter
from collections import defaultdict

In [2]:
# First we load the TF program
from tf.fabric import Fabric
from tf.app import use

In [3]:
pd.set_option('display.max_rows', None)

In [4]:
%%time
# Now we load a the BHSA database
BHS = use('etcbc/bhsa', version="2021", mod='CenterBLC/BHSaddons/tf', hoist=globals())
bhsF, bhsL, bhsT, bhsS = BHS.api.F, BHS.api.L, BHS.api.T, BHS.api.S

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,39,10938.21,100
chapter,929,459.19,100
lex,9230,46.22,100
verse,23213,18.38,100
half_verse,45179,9.44,100
sentence,63717,6.70,100
sentence_atom,64514,6.61,100
clause,88131,4.84,100
clause_atom,90704,4.70,100
phrase,253203,1.68,100


CPU times: user 2.31 s, sys: 349 ms, total: 2.66 s
Wall time: 2.78 s


## Building Text-Grammar relevant TF query

In [21]:
TextGrammarBHS='''
verse book=Reges_I chapter=17 verse*
  clause typ txt* number* rela*
   clause_atom tab* typ* pargr* number* code*
     phrase function* number*
       phrase_atom rela* number*
'''
TextGrammarBHS = BHS.search(TextGrammarBHS)
BHS.show(TextGrammarBHS, start=1, end=1, extraFeatures={'function', 'lex', 'sp', 'typ', 'ls','tab'}, condensed=False)

  0.45s 343 results


In [22]:
BHS.export(TextGrammarBHS, toDir='/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/1200_AUS-research/Fabric-TEXT/course_TF-Workshop/data_export', toFile='TextGrammarBHS.tsv')

# Loading the Search Results as `df`

In [23]:
TextGrammarBHS=pd.read_csv('/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/1200_AUS-research/Fabric-TEXT/course_TF-Workshop/data_export/TextGrammarBHS.tsv', delimiter='\t', encoding='utf-16')
pd.set_option('display.max_columns', 50)
TextGrammarBHS.head(5)

,R,S1,S2,S3,NODE1,TYPE1,TEXT1,book1,chapter1,verse1,NODE2,TYPE2,TEXT2,number2,rela2,txt2,typ2,NODE3,TYPE3,TEXT3,code3,number3,pargr3,tab3,typ3,NODE4,TYPE4,TEXT4,function4,number4,NODE5,TYPE5,TEXT5,number5,rela5
0,1,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,Reges_I,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760855,phrase,וַ,Conj,1,1020687,phrase_atom,וַ,8210,NaN
1,2,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,Reges_I,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760856,phrase,יֹּאמֶר֩,Pred,2,1020688,phrase_atom,יֹּאמֶר֩,8211,NaN
2,3,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,Reges_I,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760857,phrase,אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁבֵ֣י גִלְעָד֮,Subj,3,1020689,phrase_atom,אֵלִיָּ֨הוּ,8212,NaN
3,4,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,Reges_I,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760857,phrase,אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁבֵ֣י גִלְעָד֮,Subj,3,1020690,phrase_atom,הַתִּשְׁבִּ֜י,8213,Appo
4,5,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,Reges_I,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760857,phrase,אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁבֵ֣י גִלְעָד֮,Subj,3,1020691,phrase_atom,מִתֹּשָׁבֵ֣י גִלְעָד֮,8214,Spec


# Creating SBL conform Book Abbreviations from the BHS Latin Book Names

In [24]:
# Mapping of BHS book names to SBL abbreviations
bhs_to_sbl = {
    'Genesis': 'Gen',
    'Exodus': 'Exod',
    'Leviticus': 'Lev',
    'Numeri': 'Num',
    'Deuteronomium': 'Deut',
    'Josua': 'Josh',
    'Judices': 'Judg',
    'Ruth': 'Ruth',
    'Samuel_I': '1 Sam',
    'Samuel_II': '2 Sam',
    'Reges_I': '1 Kgs',
    'Reges_II': '2 Kgs',
    'Chronica_I': '1 Chr',
    'Chronica_II': '2 Chr',
    'Esra': 'Ezra',
    'Nehemia': 'Neh',
    'Esther': 'Esth',
    'Iob': 'Job',
    'Psalmi': 'Ps',
    'Proverbia': 'Prov',
    'Ecclesiastes': 'Eccl',
    'Canticum': 'Song',
    'Jesaia': 'Isa',
    'Jeremia': 'Jer',
    'Threni': 'Lam',
    'Ezechiel': 'Ezek',
    'Daniel': 'Dan',
    'Hosea': 'Hos',
    'Joel': 'Joel',
    'Amos': 'Amos',
    'Obadia': 'Obad',
    'Jona': 'Jonah',
    'Micha': 'Mic',
    'Nahum': 'Nah',
    'Habakuk': 'Hab',
    'Zephania': 'Zeph',
    'Haggai': 'Hag',
    'Sacharia': 'Zech',
    'Maleachi': 'Mal'
}

In [25]:
TextGrammarBHS['book1_BHS'] = TextGrammarBHS['book1']
TextGrammarBHS['book1'] = TextGrammarBHS['book1'].map(bhs_to_sbl).fillna(TextGrammarBHS['book1'])
TextGrammarBHS.sort_values(by="NODE3", ascending=True, inplace=True) # needs to be done otherwise the clause_atom order is off in the CTT file.
TextGrammarBHS.head(5)

,R,S1,S2,S3,NODE1,TYPE1,TEXT1,book1,chapter1,verse1,NODE2,TYPE2,TEXT2,number2,rela2,txt2,typ2,NODE3,TYPE3,TEXT3,code3,number3,pargr3,tab3,typ3,NODE4,TYPE4,TEXT4,function4,number4,NODE5,TYPE5,TEXT5,number5,rela5,book1_BHS
0,1,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1 Kgs,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760855,phrase,וַ,Conj,1,1020687,phrase_atom,וַ,8210,NaN,Reges_I
1,2,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1 Kgs,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760856,phrase,יֹּאמֶר֩,Pred,2,1020688,phrase_atom,יֹּאמֶר֩,8211,NaN,Reges_I
2,3,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1 Kgs,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760857,phrase,אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁבֵ֣י גִלְעָד֮,Subj,3,1020689,phrase_atom,אֵלִיָּ֨הוּ,8212,NaN,Reges_I
3,4,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1 Kgs,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760857,phrase,אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁבֵ֣י גִלְעָד֮,Subj,3,1020690,phrase_atom,הַתִּשְׁבִּ֜י,8213,Appo,Reges_I
4,5,1_Kings,17,1,1423624,verse,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1 Kgs,17,1,463599,clause,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,1,NaN,N,WayX,552846,clause_atom,וַיֹּאמֶר֩ אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁב...,0,2529,1,0,WayX,760857,phrase,אֵלִיָּ֨הוּ הַתִּשְׁבִּ֜י מִתֹּשָׁבֵ֣י גִלְעָד֮,Subj,3,1020691,phrase_atom,מִתֹּשָׁבֵ֣י גִלְעָד֮,8214,Spec,Reges_I


# Creating a CTT like Text-Grammar print

In [26]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
import pandas as pd
import re

# DATAFRAME ------------------------------------------------------------------
# Your input dataframe. This should be prepared before calling the function.
df = TextGrammarBHS  # replace with your DataFrame variable (already present)

# HEBREW / NON-HEBREW SEGMENTATION ------------------------------------------
# Regex that finds runs of Hebrew characters OR runs of non-Hebrew characters.
# We iterate these runs to set different fonts for Hebrew vs. non-Hebrew segments.
HEBREW_RE = re.compile(r'[\u0590-\u05FF\uFB1D-\uFB4F]+|[^\u0590-\u05FF\uFB1D-\uFB4F]+')

def is_hebrew_char(ch: str) -> bool:
    """Return True if the single character 'ch' is a Hebrew character (including presentation forms)."""
    code = ord(ch)
    return (0x0590 <= code <= 0x05FF) or (0xFB1D <= code <= 0xFB4F)

def split_hebrew_nonhebrew(text: str):
    """
    Generator: yields tuples (segment_text, is_hebrew_bool)
    It scans 'text' and yields consecutive runs of Hebrew or non-Hebrew.
    Use this to create runs with correct fonts in a docx paragraph.
    """
    for m in HEBREW_RE.finditer(text):
        seg = m.group(0)
        # seg[0] is safe because the regex returns at least one char
        yield seg, bool(is_hebrew_char(seg[0]))


# FONT SETTING FOR RUNS -----------------------------------------------------
def set_run_font(run, font_name: str, size_pt: float = None):
    """
    Set run font name and size in a docx run and ensure the underlying xml rFonts
    are set so Word displays the intended font for ASCII/hAnsi/cs glyph sets.
    """
    run.font.name = font_name
    if size_pt:
        run.font.size = Pt(size_pt)
    # The docx API sometimes needs explicit rFonts set on the run XML to apply fonts reliably.
    r = run._element
    rPr = r.get_or_add_rPr()
    rFonts = OxmlElement('w:rFonts')
    rFonts.set(qn('w:ascii'), font_name)
    rFonts.set(qn('w:hAnsi'), font_name)
    rFonts.set(qn('w:cs'), font_name)
    rPr.append(rFonts)


# MAIN FUNCTION --------------------------------------------------------------
def create_docx_from_grammar(df, output_file,
                             hebrew_font="SBL Biblit", hebrew_font_size=14,
                             other_font="Courier New", other_font_size=10):
    """
    Create a DOCX file with mixed Hebrew and non-Hebrew runs and hierarchical
    opening/closing separators.

    Parameters:
    - df: pandas.DataFrame with grammar rows (expected columns used below).
    - output_file: path for saving .docx
    - hebrew_font: name of font for Hebrew runs
    - hebrew_font_size: size for Hebrew runs
    - other_font: font name for everything else (metadata, separators)
    - other_font_size: size for non-Hebrew runs
    """

    # Create a new Word document
    doc = Document()

    # --- Page setup: set big page because you use wide lines -----------------
    section = doc.sections[0]
    section.page_width = Inches(25)    # very wide page to avoid line wrapping
    section.page_height = Inches(40)
    section.left_margin = Inches(0.5)
    section.right_margin = Inches(0.5)
    section.top_margin = Inches(0.5)
    section.bottom_margin = Inches(0.5)

    # --- Base 'Normal' style settings --------------------------------------
    style = doc.styles['Normal']
    style.font.name = other_font
    style.font.size = Pt(other_font_size)
    paragraph_format = style.paragraph_format
    # tight spacing so output looks compact
    paragraph_format.space_before = Pt(0)
    paragraph_format.space_after = Pt(0)
    paragraph_format.line_spacing = 1.0

    # --- Grouping: iterate over clause groups --------------------------------
    # You group by TYPE3 first, then within each TYPE3 by NODE3 (clause atoms).
    # This mirrors your earlier structure and preserves order with sort=False.
    clauseatoms = df.groupby(['TYPE3'])
    for clause_type, type_data in clauseatoms:
        clause_atoms = type_data.groupby('NODE3', sort=False)
        clauseatom_list = list(clause_atoms)

        # Unified stack to track *actual* opened separators in chronological (LIFO) order.
        # Each element is (symbol, tab_level), where symbol is one of '=', '-', '+'.
        open_stack = []

        # prev_txt holds the text_type of the previous clause atom (e.g., 'NQ', 'NQN', etc.)
        prev_txt = ""

        # --- Iterate through clause-atoms in this TYPE3 group -----------------
        for idx, (clause_id, clause_data) in enumerate(clauseatom_list):
            # Ensure rows for the clause_atom are sorted by R (word order or role order)
            clause_data = clause_data.sort_values('R')
            first_row = clause_data.iloc[0]  # representative row for metadata

            # Read metadata fields used later for line assembly
            clause_type = first_row['typ3']        # e.g., 'clause' type label
            clause_rela = first_row['rela2']      # clause relation code
            clause_atom_code = first_row['code3'] # internal code number
            text_type = first_row['txt2']         # e.g., 'N', 'NQ', 'NQN', 'NQQ', ...
            tab_level = int(first_row['tab3'])    # indentation level (int)
            clause_num = first_row['NODE3']       # node id / clause id
            paragraph = first_row['pargr3']       # paragraph id / code
            book = first_row['book1']
            chapter = first_row['chapter1']
            verse = first_row['verse1']
            dot = "."  # marker used in separator formatting

            # ---------------- Helper closures (local functions) -----------------
            # These are inside the loop because they refer to doc, open_stack, etc.

            def open_separator(symbol, tab_level, text_char, prefix="", suffix=""):
                """
                Add an opening separator line to the document and push it on open_stack.
                - symbol: one of '=', '-', '+'
                - tab_level: indentation level (used to create '  |' spacing)
                - text_char: usually '.' displayed at end-right
                - prefix/suffix: small strings like 'open>' or '+' to visualize opening
                """
                sep_spacing = "  |" * tab_level
                sep_line = f"{prefix}{symbol*80}{suffix}{sep_spacing}|{text_char:>68}" # This {text_char:>68} applies when the SBL book abbreviations are being used. Otherwise you need {text_char:>70}
                p = doc.add_paragraph(sep_line, style='Normal')
                p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
                # record the actual opening (symbol and tab) on the unified stack
                open_stack.append((symbol, tab_level))

            def close_separator(symbol, tab_level, text_char, prefix="", suffix=""):
                """
                Emit a closing separator line using the provided symbol and tab.
                This does not touch open_stack (it prints a closing line).
                """
                sep_spacing = "  |" * tab_level
                sep_line = f"{prefix}{symbol*80}{suffix}{sep_spacing}|{text_char:>68}" # This {text_char:>68} applies when the SBL book abbreviations are being used. Otherwise you need {text_char:>70}
                p = doc.add_paragraph(sep_line, style='Normal')
                p.alignment = WD_ALIGN_PARAGRAPH.RIGHT

            def close_until(target_symbols, text_char):
                """
                Pop items from open_stack and print closing separators until
                we encounter a symbol that is in target_symbols.
                This implements LIFO closings consistent with how things were opened.
                """
                while open_stack:
                    symbol, tab_level = open_stack.pop()
                    # print a closing line for this opened level
                    close_separator(symbol, tab_level, text_char, prefix="<closed", suffix="+")
                    # if the popped symbol is one of the requested targets, stop
                    if symbol in target_symbols:
                        break

            def count_diff(prev, cur):
                """
                Compute differences in counts of D, N, Q letters between two text_type strings.
                Returns a tuple (d_diff, n_diff, q_diff) where each is integer (cur - prev).
                Example: prev='NQ' cur='NQQ' -> returns (0,0,1)
                """
                return (
                    cur.count('D') - prev.count('D'),
                    cur.count('N') - prev.count('N'),
                    cur.count('Q') - prev.count('Q')
                )

            # ---------------- Compute transitions between prev and current ----------
            d_diff, n_diff, q_diff = count_diff(prev_txt, text_type)
            # d_diff > 0 : D opened, < 0 : D closed (or reduced)
            # similarly for n_diff and q_diff

            # ----------------- Opening separators (apply before clause) ----------
            # If deeper levels are opened at this clause compared to prev, do them now.
            # The order below opens D (outermost) then N then Q (innermost).
            for _ in range(d_diff if d_diff > 0 else 0):
                open_separator("+", tab_level, dot, prefix="open>", suffix="+")
            for _ in range(n_diff if n_diff > 0 else 0):
                open_separator("-", tab_level, dot, prefix="open>", suffix="+")
            for _ in range(q_diff if q_diff > 0 else 0):
                open_separator("=", tab_level, dot, prefix="open>", suffix="+")

            # ---------------- Build the phrase line (content of the clause) -------
            phraseatoms = []
            for _, row in clause_data.iterrows():
                function = row['function4']
                subphrase_rela = row['rela5']
                # take the textual content column TEXT5 (adjust if your column is different)
                text = row['TEXT5'] if row['TEXT5'] else ""
                clean_text = text.strip()
                # represent Apposition / Spec etc in a slightly different format
                if subphrase_rela in ['Appo', 'Spec', 'Para', 'Link']:
                    phraseatoms.append(f"[<{subphrase_rela}> {clean_text}< {function}>]")
                else:
                    phraseatoms.append(f"[<{function}> {clean_text}]")

            # Because Hebrew reads RTL, you used reverse ordering earlier; keep that behavior
            phraseatoms.reverse()
            spacing = "  |" * tab_level
            clause_rela_str = f"{clause_rela:<4}" if pd.notna(clause_rela) else "    "

            line_parts = (
                "".join(phraseatoms)
                + spacing
                + f"|   {tab_level:>2} {text_type:<10}{paragraph:<18}"
                  f"{clause_type:<6}{clause_rela_str} {clause_atom_code:3d} "
                  f"{clause_num:<7}{book:>5}{chapter:3d}:{verse:3d}"
            )

            # ---------------- Closing logic (hierarchical; BEFORE clause output)-
            # If this clause reduces the nesting level (some symbols close), we
            # must print the necessary closing separators before the next visible clause.
            if d_diff < 0 or n_diff < 0 or q_diff < 0:
                # Build a list of symbols to close: Q inner, N middle, D outer.
                # Example: if q_diff == -2, n_diff == -1 -> symbols_to_close = ['=', '=', '-']
                symbols_to_close = (
                    ["="] * abs(q_diff) +
                    ["-"] * abs(n_diff) +
                    ["+"] * abs(d_diff)
                )
                # We need to close innermost first, so reverse this list and pop in that order.
                for sym in reversed(symbols_to_close):
                    close_until(sym, dot)

            # ---------------- Add the phrase line to the document ----------------
            # We split the assembled line into Hebrew and non-Hebrew runs and assign fonts.
            p = doc.add_paragraph()
            p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
            for seg, is_he in split_hebrew_nonhebrew(line_parts):
                run = p.add_run(seg)
                # set appropriate font and size for each run
                set_run_font(run, hebrew_font if is_he else other_font,
                             hebrew_font_size if is_he else other_font_size)

            # Update prev_txt for the next iteration (important for diff computation)
            prev_txt = text_type

        # ---------------- Force-close any remaining open levels at end of TYPE3 ---
        # If there are leftover openings when we finish a TYPE3 group, close them in LIFO.
        while open_stack:
            symbol, tab_level = open_stack.pop()
            close_separator(symbol, tab_level, dot, prefix="<closed", suffix="+")

    # ---------------- Save the document -------------------------------------
    doc.save(output_file)
    print(f"DOCX file saved to: {output_file}")

# Example usage:
create_docx_from_grammar(
    df,
    '/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/1200_AUS-research/Fabric-TEXT/course_TF-Workshop/data_export/CTT_complex_fonts.docx'
)


DOCX file saved to: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/1200_AUS-research/Fabric-TEXT/course_TF-Workshop/data_export/CTT_complex_fonts.docx


# Quick Look at the Result without opening the docx file

In [27]:
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from IPython.display import display, HTML
import html

def display_docx_monospace_preserve_space(path, max_paragraphs=None):
    doc = Document(path)
    html_parts = []
    
    # Limit paragraphs if max_paragraphs is specified
    paragraphs = doc.paragraphs[:max_paragraphs] if max_paragraphs else doc.paragraphs
    
    for para in paragraphs:
        # Determine HTML tag (headings or normal)
        tag = "p"
        if para.style.name.startswith("Heading"):
            level = para.style.name.replace("Heading ", "")
            tag = f"h{level}"
        # Paragraph alignment
        align_map = {
            WD_ALIGN_PARAGRAPH.LEFT: "left",
            WD_ALIGN_PARAGRAPH.CENTER: "center",
            WD_ALIGN_PARAGRAPH.RIGHT: "right",
            WD_ALIGN_PARAGRAPH.JUSTIFY: "justify",
        }
        align = align_map.get(para.alignment, "left")
        # Build inner HTML with formatting
        inner_html = ""
        for run in para.runs:
            # Escape HTML special characters, then replace newlines
            text = html.escape(run.text).replace("\n", "<br>")
            run_style = "font-family: monospace;"
            if run.bold:
                run_style += "font-weight: bold;"
            if run.italic:
                run_style += "font-style: italic;"
            if run.underline:
                run_style += "text-decoration: underline;"
            inner_html += f"<span style='{run_style}'>{text}</span>"
        html_parts.append(f"<{tag} style='text-align:{align}; margin:0;'>{inner_html}</{tag}>")
    
    # Combine and display with preserved spacing
    html_output = (
        "<div style='font-family: monospace; white-space: pre-wrap; line-height: 1.0;'>"
        + "\n".join(html_parts)
        + "</div>"
    )
    display(HTML(html_output))

# Example usage:
display_docx_monospace_preserve_space("/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/1200_AUS-research/Fabric-TEXT/course_TF-Workshop/data_export/CTT_complex_fonts.docx", max_paragraphs=20)
#display_docx_monospace_preserve_space("/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/1200_AUS-research/Fabric-TEXT/course_TF-Workshop/data_export/noCTT_complex_fonts.docx", max_paragraphs=20)
